<a href="https://colab.research.google.com/github/AsimGasimov/Machine-Learning/blob/main/Resnet50_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [3]:
class ResBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()

    self.conv = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, stride, 1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 3, 1, 1),
        nn.BatchNorm2d(out_channels)
    )

    self.shortcut = nn.Sequential()
    if stride != 1 or in_channels != out_channels:
      self.shortcut = nn.Sequential(
          nn.Conv2d(in_channels, out_channels, 1, stride),
          nn.BatchNorm2d(out_channels)
      )

    self.relu = nn.ReLU()

  def forward(self, X):
    out = self.conv(X)
    out = out + self.shortcut(X)
    out = self.relu(out)
    return out

In [4]:
class ResNet(nn.Module):
  def __init__(self, n_features, out_channel):
    super().__init__()

    self.conv1 = nn.Sequential(
        nn.Conv2d(n_features, 64, 3, 1, 1),
        nn.BatchNorm2d(64),
        nn.ReLU()
    )

    self.Layer1_1 = ResBlock(64, 64)
    self.Layer1_2 = ResBlock(64, 64)

    self.Layer2_1 = ResBlock(64, 128, stride=2)
    self.Layer2_2 = ResBlock(128, 128)

    self.Layer3_1 = ResBlock(128, 256, stride=2)
    self.Layer3_2 = ResBlock(256, 256)

    self.Layer4_1 = ResBlock(256, 512, stride=2)
    self.Layer4_2 = ResBlock(512, 512)

    self.avgpool = nn.AdaptiveAvgPool2d((1,1))
    self.fc = nn.Linear(512, out_channel)

  def forward(self, X):
    X = self.conv1(X)

    X = self.Layer1_1(X)
    X = self.Layer1_2(X)

    X = self.Layer2_1(X)
    X = self.Layer2_2(X)

    X = self.Layer3_1(X)
    X = self.Layer3_2(X)

    X = self.Layer4_1(X)
    X = self.Layer4_2(X)

    X = self.avgpool(X)
    X = torch.flatten(X, 1)
    X = self.fc(X)

    return X

In [5]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_data = datasets.CIFAR10(root='./data', train=True,
                              download=True, transform=transform)

valid_data = datasets.CIFAR10(root='./data', train=False,
                              download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=128)

100%|██████████| 170M/170M [00:05<00:00, 33.8MB/s]


In [ ]:
if torch.cuda.is_available():
  device='cuda'
elif torch.backends.mps.is_available():
  device='mps'
else:
  device='cpu'

In [17]:


model = ResNet(3, 10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [7]:
def eval(model, criterion, valid_loader, device):
  valid_loss = 0.
  model.eval()

  with torch.no_grad():
    for image, label in valid_loader:
      image, label = image.to(device), label.to(device)

      preds = model(image)
      loss = criterion(preds, label)

      valid_loss += loss.item()

  return valid_loss / len(valid_loader)

In [8]:
def train(model, criterion, optimizer, train_loader,
          valid_loader, scheduler, n_epochs, device):

  for epoch in range(n_epochs):
    train_loss = 0.
    model.train()
    current_lr = optimizer.param_groups[0]['lr']

    for image, label in train_loader:
      image, label = image.to(device), label.to(device)

      preds = model(image)
      loss = criterion(preds, label)

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

      train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_valid_loss = eval(model, criterion, valid_loader, device)

    scheduler.step()

    print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_valid_loss:.4f} | LR: {current_lr}")

In [9]:
train(model, criterion, optimizer,
      train_loader, valid_loader,
      scheduler, n_epochs=10, device=device)

Epoch 1/10 | Train Loss: 1.3423 | Val Loss: 1.2307 | LR: 0.001
Epoch 2/10 | Train Loss: 0.8170 | Val Loss: 0.8316 | LR: 0.001
Epoch 3/10 | Train Loss: 0.5997 | Val Loss: 0.9255 | LR: 0.001
Epoch 4/10 | Train Loss: 0.4617 | Val Loss: 0.6043 | LR: 0.001
Epoch 5/10 | Train Loss: 0.3703 | Val Loss: 0.7209 | LR: 0.001
Epoch 6/10 | Train Loss: 0.1743 | Val Loss: 0.4206 | LR: 0.0001
Epoch 7/10 | Train Loss: 0.1126 | Val Loss: 0.4368 | LR: 0.0001
Epoch 8/10 | Train Loss: 0.0767 | Val Loss: 0.4724 | LR: 0.0001
Epoch 9/10 | Train Loss: 0.0493 | Val Loss: 0.5093 | LR: 0.0001
Epoch 10/10 | Train Loss: 0.0285 | Val Loss: 0.5481 | LR: 0.0001
